In [1]:
from __future__ import annotations

# load useful packages
import xarray as xr
import xesmf as xe
import numpy as np
xr.set_options(display_style='html')
import cftime
import matplotlib.pyplot as plt

In [2]:
def consistent_naming(_ds):
    """
    The naming convention for coordinates and dimensions are not the same 
    for noresm raw output and cmorized variables. This function rewrites the 
    coords and dims names to be consistent and the functions thus work on all
    Choose the cmor naming convention.

    Parameters
    ----------
   _ds : xarray.Dataset 

    Returns
    -------
   _ds : xarray.Dataset

    """
    #print(_ds)
    if "latitude" in _ds.coords and "lat" not in _ds.coords:
       _ds =_ds.swap_dims({"latitude": "lat"})
    if "longitude" in _ds.coords and "lon" not in _ds.coords:
       _ds =_ds.rename({"longitude": "lon"})
    if "i" in _ds.dims:
       _ds =_ds.swap_dims({"i": "x"})
    if "j" in _ds.dims:
       _ds =_ds.swap_dims({"j": "y"})
    if "ni" in _ds.dims:
       _ds =_ds.swap_dims({"ni": "x"})
    if "nj" in _ds.dims:
       _ds =_ds.swap_dims({"nj": "y"})
    if "nlat" in _ds.dims:
       _ds =_ds.swap_dims({"nlat": "y"})
    if "nlon" in _ds.dims:
       _ds =_ds.swap_dims({"nlon": "x"})
    if "nv" in _ds.dims:
       _ds =_ds.swap_dims({"nv": "vertices"})
    if "TLAT" in _ds.coords:
       _ds =_ds.rename({"TLAT": "lat"})
    if "TLONG" in _ds.coords:
       _ds =_ds.rename({"TLONG": "lon"})
    if "depth" in _ds.dims:
       _ds =_ds.swap_dims({"depth": "lev"})
    if "depth_bnds" in _ds.dims:
       _ds =_ds.swap_dims({"depth_bnds": "lev_bnds"})
    if "nbnd" in _ds.dims:
       _ds =_ds.swap_dims({"nbnd": "bnds"})
    if "hist_interval" in _ds.dims:
       _ds =_ds.swap_dims({"hist_interval": "bnds"})
    if "nbounds" in _ds.dims:
       _ds =_ds.swap_dims({"nbounds": "bnds"})
    if "bounds" in _ds.dims:
       _ds =_ds.swap_dims({"bounds": "bnds"})
    if "type" in _ds.coords:
       _ds =_ds.drop_vars("type")
    if 'latitude_bnds' in _ds.variables:
       _ds =_ds.swap_dims({'latitude_bnds':'lat_bnds'})
    if 'longitude_bnds' in _ds.variables:
       _ds =_ds.swap_dims({'longitude_bnds':'lon_bnds'})
    if 'nav_lat' in _ds.coords:
       _ds =_ds.rename({'nav_lon':'lon','nav_lat':'lat'})
    return _ds

In [3]:
def make_latlon_bounds(ds):
    """
    This function creates latitude and longitude boundaries from lat and lon

    Parameters
    ----------
    ds : xarray.DataSet, with longitude (lon) and latitude (lat) information

    Returns
    -------
    ds:  xarray.DataSet, returns the same xarray.Dataset with lon/lat bounds added
    """
    lon_b = np.concatenate(
        (
            np.array([ds.lon[0].values - 0.5 * ds.lon.diff(dim="lon").values[0]]),
            0.5 * ds.lon.diff(dim="lon").values + ds.lon.values[:-1],
            np.array([ds.lon[-1].values + 0.5 * ds.lon.diff(dim="lon").values[-1]]),
        )
    )
    lon_b = np.reshape(np.concatenate([lon_b[:-1], lon_b[1:]]), [2, len(ds.lon.values)]).T
    lon_b = xr.DataArray(lon_b, dims=("lon", "bnds"), coords={"lon": ds.lon})
    lon_b.attrs["units"] = "degrees_east"
    lon_b.attrs["axis"] = "X"
    lon_b.attrs["bounds"] = "lon_bnds"
    lon_b.attrs["standard_name"] = "longitude_bounds"
    lon_b.attrs["long_name"] = "Longitude bounds"
    ds["lon_bnds"] = lon_b
    lat_b = np.concatenate(
        (np.array([-90], float), 0.5 * ds.lat.diff(dim="lat").values + ds.lat.values[:-1], np.array([90], float))
    )
    lat_b = np.reshape(np.concatenate([lat_b[:-1], lat_b[1:]]), [2, len(ds.lat.values)]).T
    lat_b = xr.DataArray(lat_b, dims=("lat", "bnds"), coords={"lat": ds.lat})
    lat_b = lat_b.where(lat_b < 90.0, 90.0)
    lat_b = lat_b.where(lat_b > -90.0, -90.0)
    lat_b.attrs["long_name"] = "Latitude bounds"
    lat_b.attrs["units"] = "degrees_north"
    lat_b.attrs["axis"] = "Y"
    lat_b.attrs["bounds"] = "lat_bnds"
    lat_b.attrs["standard_name"] = "latitude_bounds"
    ds["lat_bnds"] = lat_b
    return ds

In [4]:
def make_outgrid(ds: xr.Dataset, curvilinear: bool=True )-> xr.Dataset:
    """Using xesmf output grids can easily be generated by the use of e.g xe.util.grid_2D, xe.util.grid_global
    But if a specific grid is preferable, it can be generated by the use of this function

    Parameters
    ----------
    ds :         xarray.Dataset with grid information on 1D lat/lon grid
    curvilinear: bool . Default is True. Set to False for rectilinear grids

    Returns
    -------
    outgrid : xarray.DataSet with output grid information to be used for regridding files
    """
    if not curvilinear:
        outgrid = xr.Dataset(
    {
        "lat": ds.lat,
        "lon": ds.lon,
    })

    else:
        outgrid = xr.Dataset()
        if "lon_bnds" not in ds.variables or "lat_bnds" not in ds.variables:
            ds = make_latlon_bounds(ds)
        if "lat" in ds.dims and "lon" in ds.dims:
            ds = ds.rename_dims({"lat":"y", "lon":"x"})
        lat = xr.DataArray(np.tile(ds.lat.values, (len(ds.lon.values),1)).T, dims=("y","x"))
        outgrid = outgrid.assign_coords(lat=lat)
        lon = xr.DataArray(np.tile(ds.lon.values, (len(ds.lat.values),1)), dims=("y","x"))
        outgrid = outgrid.assign_coords(lon=lon)
        if "lat" in ds.lat_bnds.coords:
            lat_b = (
                xr.concat([ds.lat_bnds.isel(bnds=0).drop_vars("lat"), ds.lat_bnds.isel(y=-1).isel(bnds=1).drop_vars("lat")], dim="y")
                .rename("lat_b").swap_dims({ "y": "y_b"})
            )
        else:
            lat_b = (
                xr.concat([ds.lat_bnds.isel(bnds=0), ds.lat_bnds.isel(y=-1).isel(bnds=1)], dim="y")
                .rename("lat_b").swap_dims({ "y": "y_b"})
            )
        if "lon" in ds.lon_bnds.coords:
            lon_b = (
                xr.concat([ds.lon_bnds.isel(bnds=0).drop_vars("lon"), ds.lon_bnds.isel(x=-1).isel(bnds=1).drop_vars("lon")], dim="x")
                .rename("lon_b").swap_dims({"x": "x_b"})
            )
        else:
            lon_b = (
                xr.concat([ds.lon_bnds.isel(bnds=0), ds.lon_bnds.isel(x=-1).isel(bnds=1)], dim="x")
                .rename("lon_b").swap_dims({"x": "x_b"})
            )
        lat_b = xr.DataArray(np.tile(lat_b.values, (len(lon_b.values),1)).T, dims=("y_b","x_b"))
        lon_b = xr.DataArray(np.tile(lon_b.values, (len(lat_b.values),1)), dims=("y_b","x_b"))
        outgrid = outgrid.assign_coords(lat_b=lat_b)
        outgrid = outgrid.assign_coords(lon_b=lon_b)      
    return outgrid


In [5]:
def make_bounds_ocean(_ds):
    """
    This function calculates latitude and longitude values and boundaries used for regridding
    from curvilinear grids to regular lat/lon grid (rectilinear grid)
    The Dataset generated is used as_ds_in in the regridder function

    The ocean/sea-ice grid of NorESM2 is a tripolar grid with 360 and 384 unique grid cells in i- and j-direction, respectively.
    Due to the way variables are staggered in the ocean model, an additional j-row is required explaining the 385 grid cells in the j-direction for the ocean grid.
    The row with j=385 is a duplicate of the row with j=384, but with reverse i-index.
    Ocean variables are on i=360, j=385 grid
    Sea-ice variables are on i=360, j=384 grid

    Parameters
    ----------
   _ds : xarray.DataSet, with model grid information for the data which need to be regridded
                         on (i,j,vertices) format where the 4 vertices give grid corner information

    Returns
    -------
   _ds_in :  xarray.DataSet, with 2D model grid information for the data which need to be regridded
    """
 
    ny, nx =_ds.lat.shape  # ny will be 384 for sea-ice variables
    if "lat"in _ds.lon.coords:
        lat_model =_ds.lat.isel(y=slice(0, ny)).drop_vars("lon").drop_vars("lat")
        lon_model =_ds.lon.isel(y=slice(0, ny)).drop_vars("lon").drop_vars("lat")
    else:
        lat_model =_ds.lat.isel(y=slice(0, ny))
        lon_model =_ds.lon.isel(y=slice(0, ny))
    lon_b_model = xr.concat(
        [_ds.vertices_longitude.isel(vertices=0),_ds.vertices_longitude.isel(vertices=1, x=-1)], dim="x"
    )
    
    lon_b_model = xr.concat(
        [
            lon_b_model,
            xr.concat(
                [_ds.vertices_longitude.isel(vertices=3, y=-1),_ds.vertices_longitude.isel(vertices=2, y=-1, x=-1)],
                dim="x",
            ),
        ],
        dim="y",
    )
    lat_b_model = xr.concat(
        [_ds.vertices_latitude.isel(vertices=0),_ds.vertices_latitude.isel(vertices=1, x=-1)], dim="x"
    )
    lat_b_model = xr.concat(
        [
            lat_b_model,
            xr.concat(
                [_ds.vertices_latitude.isel(vertices=3, y=-1),_ds.vertices_latitude.isel(vertices=2, y=-1, x=-1)],
                dim="x",
            ),
        ],
        dim="y",
    )
    if "lat" in lon_b_model.coords:
        lon_b_model = (
            lon_b_model.isel(y=slice(0, ny + 1))
            .rename("lon_b")
            .swap_dims({"y": "y_b", "x": "x_b"})
            .drop_vars("lon")
            .drop_vars("lat")
        )
        lat_b_model = (
            lat_b_model.isel(y=slice(0, ny + 1))
            .rename("lat_b")
            .swap_dims({"y": "y_b", "x": "x_b"})
            .drop_vars("lon")
            .drop_vars("lat")
        )
    else:
        lon_b_model = lon_b_model.isel(y=slice(0, ny + 1)).rename("lon_b").swap_dims({"y": "y_b", "x": "x_b"})
        lat_b_model = lat_b_model.isel(y=slice(0, ny + 1)).rename("lat_b").swap_dims({"y": "y_b", "x": "x_b"})
    lat_b_model = lat_b_model.where(lat_b_model < 90.0, 90.0)
    lat_b_model = lat_b_model.where(lat_b_model > -90.0, -90.0)
    
    if "time" in lat_b_model.dims:
        lat_b_model = lat_b_model.isel(time=0).drop_vars("time")
        lon_b_model = lon_b_model.isel(time=0).drop_vars("time")
   
    if "month" in lat_b_model.dims:
        lat_b_model = lat_b_model.isel(month=0).drop_vars("month")
        lon_b_model = lon_b_model.isel(month=0).drop_vars("month")
 
    ds_out = xr.Dataset()
    ds_out= ds_out.assign_coords(lat=lat_model)
    ds_out = ds_out.assign_coords(lon=lon_model)  
    ds_out= ds_out.assign_coords(lat_b=lat_b_model)
    ds_out = ds_out.assign_coords(lon_b=lon_b_model)
 
    keep_coords = ["lat", "lon", "lat_b", "lon_b"]
    for coord in list(ds_out.coords):
        if coord not in keep_coords:
            ds_out = ds_out.drop_vars(coord)
     
    return ds_out


In [ ]:
def make_regridder(
   _ds, outgrid, regrid_mode="conservative", curvilinear=True, periodic = True
):
    """The first step of the regridding routine!
    There is an important reason why the regridding is broken into two steps
    (making the regridder and perform regridding). For high-resolution grids,
    making the regridder (i.e. “computing regridding weights”) is quite computationally expensive,
    but performing regridding on data (“applying regridding weights”) is still pretty fast.

    Parameters
    ----------
   _ds :                 xarray.DataSet, with model grid information for the data which need to be regridded
    outgrid :            xarray.DataSet, with output grid information which the data will be regridded to
    regrid_mode : str,   'bilinear', 'conservative', 'patch', 'nearest_s2d', 'nearest_d2s'
    curvilinear :         bool, True for ocean and se-ice variables. False for atmosphere

    A comment about the ignore_degenerate = True option: if the grid cells have very close corners, the ESMF thinks that
    it is a triangle instead of a quadrilateral (i.e. "degenerated") and throws out an error which can be skipped by 
    setting ignore_degenerate=True. Alternatively, you can manually remove such degenerated cells.
    
    Returns
    -------
    regridder : xarray.DataSet with regridder weight file information
    """
    ds_in = make_bounds_ocean(_ds)
    regridder = xe.Regridder(ds_in, outgrid, method=regrid_mode, periodic=periodic, ignore_degenerate=True)
    return regridder


In [7]:
def regrid_file(
   _ds, var, outgrid,  grid_weight_path=None, regrid_mode="conservative", curvilinear=True, periodic = True
):
    """Second step of the regridding routine!

    Parameters
    ----------
   _ds :                 xarray.DataSet, with model grid information for the data which need to be regridded
    var :                str, name of varible
    outgrid :            xarray.DataSet, with output grid information which tif 'plev'in _ds[var].dims:
               
    Returns
    -------
    dr : xarray.Dataset with regridded variable data and lon from 0-360
    """
    print("In regrid_file")
    regridder= make_regridder(_ds, outgrid, regrid_mode, curvilinear, periodic)
    print("Regridding maker made")
    # Fix for numpy 2 while waiting for xESMF 0.8.7
    regridder.shape_in = tuple(map(int, regridder.shape_in))
    regridder.shape_out = tuple(map(int, regridder.shape_out))
    dr = regridder(_ds[var])  # needs DataArray
    lon = dr.lon.isel(y=0).drop_vars('lat').drop_vars('lon')
    lat = dr.lat.isel(x=0).drop_vars('lat').drop_vars('lon')
    dr = dr.drop_vars('lat').drop_vars('lon')
    dr = dr.assign_coords(lat=lat)
    dr = dr.assign_coords(lon=lon)
    dr = dr.swap_dims({"x": "lon", "y": "lat"})
    dr = dr.to_dataset(name=var)
    if "long_name" in _ds[var].attrs:
        dr[var].attrs["long_name"] =_ds[var].long_name
    if "units" in _ds[var].attrs:
        dr[var].attrs["units"] =_ds[var].units
    if "standard_name"in _ds[var].attrs:
        dr[var].attrs["standard_name"] =_ds[var].standard_name
    print("Regridding completed")
    return dr


In [12]:
def regrid_ocean(_da,var:str, modelversion:str, outdir: str):#, expid: str, first_year:int, last_year:int,  tempstorage: str = "/scratch/adagj/noresm_raw/temp/" ):
    # REGRID info
    _ds = _da.squeeze().to_dataset(name=var)
    _ds = consistent_naming(_ds)
    area = xr.open_dataset(
        "/projects/NS9034K/CMIP6/CMIP/NCC/NorESM2-MM/piControl/r1i1p1f1/fx/areacella/gn/latest/areacella_fx_NorESM2-MM_piControl_r1i1p1f1_gn.nc"
    )
    outgrid = make_outgrid(area)
    
    if modelversion == "NorESM2":
        grid = xr.open_dataset("/nird/home/adagj/pyclim-adagj/ocean/grid_tnx1v4_20170601.nc")
        
    if modelversion in ["NorESM1", "CESM1"]:
        grid = xr.open_dataset("/nird/home/adagj/pyclim-adagj/ocean/grid_gx1v6_20101119.nc")
    
    grid = consistent_naming(grid)
    vertices_latitude = grid.pclat.to_dataset(name = "vertices_latitude")
    vertices_longitude = grid.pclon.to_dataset(name = "vertices_longitude")
    _ds = xr.merge([_ds,vertices_latitude,vertices_longitude]) 
    if modelversion == "NorESM2":
        _lat = grid.plat.to_dataset(name = "lat")
        _lon = grid.plon.to_dataset(name = "lon")
        _ds = xr.merge([_ds,_lat,_lon]) 



    _ds = xr.merge([_ds,vertices_latitude,vertices_longitude]) 
    grid_weight_path = outdir
    regrid_mode = "nearest_s2d"

    dr = regrid_file(
                _ds,
                var=var,
                outgrid=outgrid,
                grid_weight_path=grid_weight_path,
                regrid_mode=regrid_mode,
                curvilinear=True,
                periodic=True,
            )
    return dr

In [9]:
def test_regridding(ds, dr, var:str = "sst", modelversion: str = "NorESM2"):
    da = consistent_naming(ds[var].to_dataset(name=var))
    if modelversion == "NorESM2":
        grid = xr.open_dataset("/nird/home/adagj/pyclim-adagj/ocean/grid_tnx1v4_20170601.nc")
        
    if modelversion in ["NorESM1", "CESM1"]:
        grid = xr.open_dataset("/nird/home/adagj/pyclim-adagj/ocean/grid_gx1v6_20101119.nc")
    
    grid = consistent_naming(grid)
    parea = grid.parea*grid.pmask
    parea = parea.assign_coords({"x":da.x, "y":da.y})
    parea = parea.where(parea>0)
    gmsst = (da[var].isel(time=0)*parea).sum(dim=("x","y"))/parea.sum(dim=("x","y"))
    print(f"Global average {var} raw data: {gmsst.values} degC")
    
    area = xr.open_dataset(
        "/projects/NS9034K/CMIP6/CMIP/NCC/NorESM2-MM/piControl/r1i1p1f1/fx/areacella/gn/latest/areacella_fx_NorESM2-MM_piControl_r1i1p1f1_gn.nc"
    )
    mask = xr.open_dataset("/projects/NS9034K/CMIP6/CMIP/NCC/NorESM2-MM/piControl/r1i1p1f1/fx/sftlf/gn/latest/sftlf_fx_NorESM2-MM_piControl_r1i1p1f1_gn.nc")
    mask = 1-1e-2*mask.sftlf
    mask = mask.assign_coords({"lat":area.lat, "lon":area.lon})
    ocnarea = area.areacella*mask
    ocnarea = ocnarea.assign_coords({"lat":dr.lat, "lon":dr.lon})
    ocnarea = ocnarea.where(ocnarea>0)
    gmsstrg =  (dr[var]*ocnarea).sum(dim=("lat","lon"))/ocnarea.sum(dim=("lat","lon"))

    print(f"Global average {var} regridded data: {gmsstrg.values} degC")

In [ ]:
#testing NorESM1 / CESM1 data running on gx1v6 grid
ds = xr.open_dataset("/nird/datalake/NS9039K/users/ingo/data/archive/B20TRLENS_nudgeEra5AnomUVPS6h_s19500101_03/ocn/hist/B20TRLENS_nudgeEra5AnomUVPS6h_s19500101_03.pop.h.1980-01.nc") 
var = "SST"
modelversion = "CESM1"
dr = regrid_ocean(ds[var], var, modelversion, "/scratch/adagj/noresm_raw/temp/")
test_regridding(ds, dr, var, modelversion)

In regrid_file
Making bounds
The bounds are made!


/projects/NS10013K/adagj/miniconda3/envs/xesmf_env/lib/python3.10/site-packages/xesmf/backend.py:41: UserWarning: Input array is not F_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not F_CONTIGUOUS. ' 'Will affect performance.')


Regridding maker made
Regridding completed
Global average SST raw data: 18.372363952111364 degC
Global average SST regridded data: 18.13496971130371 degC


In [11]:
#testing NorESM2 running on tnx1v4 grid
ds = xr.open_dataset("/nird/datalake/NS9560K/noresm/cases/N1850frc2_f09_tn14_20191001/ocn/hist/N1850frc2_f09_tn14_20191001.micom.hm.1200-01.nc") 
var = "sst"
modelversion = "NorESM2"
dr = regrid_ocean(ds[var], var, modelversion, "/scratch/adagj/noresm_raw/temp/")
test_regridding(ds, dr, var, modelversion)

<xarray.Dataset> Size: 12MB
Dimensions:             (y: 385, x: 360, vertices: 4)
Coordinates:
    time                object 8B 1200-01-17 00:00:00
Dimensions without coordinates: y, x, vertices
Data variables:
    sst                 (y, x) float32 554kB ...
    vertices_latitude   (vertices, y, x) float64 4MB -80.2 -80.2 ... 62.47 62.23
    vertices_longitude  (vertices, y, x) float64 4MB -110.0 -109.0 ... -109.9
    lat                 (y, x) float64 1MB ...
    lon                 (y, x) float64 1MB ...
In regrid_file
Making bounds
The bounds are made!


/projects/NS10013K/adagj/miniconda3/envs/xesmf_env/lib/python3.10/site-packages/xesmf/backend.py:41: UserWarning: Input array is not F_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not F_CONTIGUOUS. ' 'Will affect performance.')


Regridding maker made
Regridding completed
Global average sst raw data: 18.551435266687694 degC
Global average sst regridded data: 18.374462127685547 degC
